# **NanoGPTによる潰滅的忘却演習**

## **ポイント**

青空文庫の中島敦コーパスで事前学習した小型 GPT に、脳科学用語の Wikipedia で追加学習すると、新ドメインの validation loss は大幅に低下する一方、旧ドメインの loss は上昇した。生成も旧文体から科学用語中心へと変化した。これは継続学習における壊滅的忘却(Catasrtrophic Forgetting)の簡易的な再現である。

このコードは、2026年度、東京科学大学、大学院文系教養科目、「言語と身体」の講義用の参考資料として作成されたものである。

## **資料**

**1. NanoGPT**

https://nano-gpt.com/api

元OpenAIの研究者であるアンドレイ・カーパシー氏が公開した教育・研究用の最小限のGPT実装プロジェクト

**2.nanoGPTで５分で終わるLLMの事前学習をしてみました**

https://www.dmacs.net/blog/nanogptgohun001/

マッキーISOさんによるコードで青空文庫のコーパスを利用した事前学習法を踏襲させていただいた。

**3.生成AI、Grokによるプログラミング支援**

## **注意**

Google Colabで実行することを前提にしています。
注意点は以下。

ランタイムを GPU に切り替える（最重要）メニュー → 「ランタイム」 → 「ランタイムのタイプを変更」

ハードウェア アクセラレータ を 「T4 GPU」 に変更

「保存」 をクリック

# **補足**

本実験は旧タスク性能の低下（CF）を示す。新規タスク学習力の低下（plasticity loss）とは指標が異なる。

本実験の条件は以下の通り。

モデル 1.81M、block 128、batch 16、accum 1、A 5000 step、B 追加 2000 step（通算 7000）、dropout 0.1、共有 SPM vocab 8000



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# -*- coding: utf-8 -*-
# nanoGPT 継続学習デモ：環境 + データ準備（A:Aozora中島敦 / B:Wiki脳科学）

import os
import re
import json
import time
import math
import pickle
import zipfile
import pathlib
import shutil
import urllib.request
import urllib.parse
import urllib.error
import numpy as np


In [ ]:
# ============================================================
# 1. 作業ディレクトリ & nanoGPT
# ============================================================
os.chdir("/content")
NANOGPT_DIR = "/content/nanoGPT"

if not os.path.exists(NANOGPT_DIR):
    print("=== nanoGPT をクローン ===")
    !git clone https://github.com/karpathy/nanoGPT.git
else:
    print("✅ nanoGPT は既に存在します")

os.chdir(NANOGPT_DIR)
print("cwd:", os.getcwd())


=== nanoGPT をクローン ===
Cloning into 'nanoGPT'...
remote: Enumerating objects: 689, done.
remote: Total 689 (delta 0), reused 0 (delta 0), pack-reused 689 (from 1)
Receiving objects: 100% (689/689), 981.25 KiB | 3.65 MiB/s, done.
Resolving deltas: 100% (380/380), done.
cwd: /content/nanoGPT


In [ ]:
# ============================================================
# 2. GPU & 依存関係（torchは入れない）
# ============================================================
print("=== GPUチェック ===")
gpu_info = os.popen("nvidia-smi").read()
if "failed" in gpu_info.lower() or not gpu_info.strip():
    raise RuntimeError("GPUがありません。ランタイムをT4 GPUに変更してください。")
print(gpu_info)

!pip -q install numpy transformers datasets tiktoken sentencepiece

import torch
import sentencepiece as spm

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("CUDA不可。GPUランタイムか確認してください。")
print(f"GPU: {torch.cuda.get_device_name(0)}")
device = "cuda"


=== GPUチェック ===
Wed Sep 16 05:57:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   38C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------------------

In [ ]:
# ============================================================
# 3. パス定義（ここを統一）
# ============================================================
DIR_A  = pathlib.Path("data/nakajimaton_aozora_bpe")   # コーパスA
DIR_B  = pathlib.Path("data/neuro_wiki_bpe")            # コーパスB
DIR_AB = pathlib.Path("data/nakajimaton_neuro_bpe")     # A+B & 共有spm
TMP    = pathlib.Path("data/_tmp_nakajima_neuro")
for d in [DIR_A, DIR_B, DIR_AB, TMP]:
    d.mkdir(parents=True, exist_ok=True)

OUT_A = "out-A"
OUT_A_THEN_B = "out-A-then-B"

# 学習ハイパーパラメータ（後で学習セルでも使用）
TRAIN_ARGS_COMMON = dict(
    device=device,
    compile=False,
    eval_interval=200,
    eval_iters=20,
    log_interval=20,
    max_iters=5000,
    lr_decay_iters=5000,
    warmup_iters=2000,
    block_size=128,
    batch_size=16,
    n_layer=4,
    n_head=4,
    n_embd=128,
    dropout=0.1,
    wandb_log=False,
)

PROMPTS_A = ["孔子は思わず", "漢の武帝の", "今から一年程前"]
PROMPTS_B = ["前頭葉は", "ドーパミン受容体は", "線条体は"]


In [ ]:
# ============================================================
# 4-1. コーパスA：中島敦
# 青空文庫に収められた中島敦の全作品(旧仮名遣いは除く)
# ============================================================
print("\n=== 4-1. コーパスA（中島敦）===")

works_A = [
    ("riryo", "https://www.aozora.gr.jp/cards/000119/files/1737_ruby_5656.zip"),	# 李陵
    ("sangetsuki","https://www.aozora.gr.jp/cards/000119/files/624_ruby_5668.zip"), # 山月記
    ("rousitsuki","https://www.aozora.gr.jp/cards/000119/files/42301_ruby_16175.zip"), # 狼疾記
    ("desi","https://www.aozora.gr.jp/cards/000119/files/1738_ruby_16462.zip"), # 弟子
    ("hikaritokazetoyume","https://www.aozora.gr.jp/cards/000119/files/1743_ruby_5730.zip"), # 光と風と夢
    ("eikyo", "https://www.aozora.gr.jp/cards/000119/files/24438_ruby_11131.zip") ,	#盈虚
    ("kansyo", "https://www.aozora.gr.jp/cards/000119/files/46429_ruby_26935.zip"), # 環礁
    ("kitsunetsuki", "https://www.aozora.gr.jp/cards/000119/files/56247_ruby_72246.zip"), # 狐憑
    ("gyujin", "https://www.aozora.gr.jp/cards/000119/files/1742_ruby_5585.zip"), # 牛人
    ("kyoukasinobunsyo", "https://www.aozora.gr.jp/cards/000119/files/24441_ruby_11129.zip"), # 鏡花氏の文章
    ("kyoutikutonoienoonna", "https://www.aozora.gr.jp/cards/000119/files/4879_ruby_11348.zip"), # 夾竹桃の家の女
    ("gojyousyusse", "https://www.aozora.gr.jp/cards/000119/files/2521_ruby_5328.zip"), # 悟浄出世
    ("jyunen", "https://www.aozora.gr.jp/cards/000119/files/58014_ruby_61294.zip"), # 十年
    ("setonaouji", "https://www.aozora.gr.jp/cards/000119/files/56244_ruby_53024.zip"), # セトナ皇子
    ("takonokinositade", "https://www.aozora.gr.jp/cards/000119/files/56242_ruby_53019.zip"), # 章魚木の下で
    ("tonansensei", "https://www.aozora.gr.jp/cards/000119/files/1741_ruby_17142.zip"), # 斗南先生
    ("toragari", "https://www.aozora.gr.jp/cards/000119/files/24439_ruby_11130.zip"), # 虎狩
    ("nantoutankouhuku", "https://www.aozora.gr.jp/cards/000119/files/619_ruby_2307.zip"), # 南島譚01幸福
    ("nantoutanhuuhu", "https://www.aozora.gr.jp/cards/000119/files/43044_ruby_16211.zip"), # 南島譚02夫婦
    ("nantoutantori", "https://www.aozora.gr.jp/cards/000119/files/43045_ruby_16212.zip"), # 南島譚03 雞
    ("purunowakide", "https://www.aozora.gr.jp/cards/000119/files/58575_ruby_64549.zip"), # プウルの傍で
    ("miira", "https://www.aozora.gr.jp/cards/000119/files/56248_ruby_73153.zip"), # 木乃伊
    ("meijinden", "https://www.aozora.gr.jp/cards/000119/files/621_ruby_661.zip"), # 名人伝
    ("mojika", "https://www.aozora.gr.jp/cards/000119/files/622_ruby_14496.zip"), # 文字禍
    ("youhunroku", "https://www.aozora.gr.jp/cards/000119/files/56243_ruby_53023.zip"), # 妖氛録
]

def clean_aozora_text(text: str) -> str:
    text = re.sub(r"《.*?》", "", text)
    text = re.sub(r"［＃.*?］", "", text)
    text = re.sub(r"｜", "", text)
    text = re.sub(r"[ 　]+\n", "\n", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

texts_A = []
for name, url in works_A:
    zip_path = TMP / f"{name}.zip"
    if not zip_path.exists():
        print(" downloading A:", url)
        urllib.request.urlretrieve(url, zip_path)
    with zipfile.ZipFile(zip_path, "r") as z:
        txt_names = [n for n in z.namelist() if n.lower().endswith(".txt")]
        if not txt_names:
            raise RuntimeError(f"txt not found in {zip_path}")
        raw = z.read(txt_names[0])
        try:
            s = raw.decode("shift_jis")
        except UnicodeDecodeError:
            s = raw.decode("utf-8", errors="ignore")
        texts_A.append(clean_aozora_text(s))

data_A = "\n\n".join(texts_A)
print(f"✅ コーパスA: {len(data_A):,} 文字")



=== 4-1. コーパスA（中島敦）===
 downloading A: https://www.aozora.gr.jp/cards/000119/files/1737_ruby_5656.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/624_ruby_5668.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/42301_ruby_16175.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/1738_ruby_16462.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/1743_ruby_5730.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/24438_ruby_11131.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/46429_ruby_26935.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/56247_ruby_72246.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/1742_ruby_5585.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/24441_ruby_11129.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/4879_ruby_11348.zip
 downloading A: https://www.aozora.gr.jp/cards/000119/files/2521_ruby_5328.zip
 downloading A: h

In [ ]:
# ============================================================
# 4-2. コーパスB：Wikipedia 脳科学
# HTTP Error 429等で取得が断念されたファイルについては、再度
# このセルを何回か実行してください。既にダウンロードされたものは
# キャッシュしてあるのでスキップされます。
# ============================================================
print("\n=== 4-2. コーパスB（Wikipedia）===")

def clean_wikipedia_text(text: str) -> str:
    if not text:
        return ""
    text = re.sub(r"\[\d+\]", "", text)
    text = re.sub(r"\[要出典\]", "", text)
    text = re.sub(r"\[誰?\]", "", text)
    text = re.sub(r"\[いつ?\]", "", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    lines = [ln.strip() for ln in text.splitlines() if len(ln.strip()) >= 2]
    return "\n".join(lines).strip()

def fetch_wikipedia_text(title: str, lang: str = "ja", max_retries: int = 3) -> str:
    params = {
        "action": "query",
        "prop": "extracts",
        "explaintext": "1",
        "exsectionformat": "plain",
        "redirects": "1",
        "format": "json",
        "titles": title,
    }
    url = f"https://{lang}.wikipedia.org/w/api.php?" + urllib.parse.urlencode(params)
    headers = {
        "User-Agent": "nanoGPT-continual-learning-demo/1.0 (educational; contact: local-colab-user)"
    }
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            req = urllib.request.Request(url, headers=headers)
            with urllib.request.urlopen(req, timeout=30) as res:
                payload = json.loads(res.read().decode("utf-8"))
            pages = payload.get("query", {}).get("pages", {})
            for _, page in pages.items():
                if "extract" in page and page["extract"]:
                    return page["extract"]
                if page.get("missing") is not None:
                    print(f"  ⚠️ ページなし: {title}")
                    return ""
            return ""
        except urllib.error.HTTPError as e:
            last_err = e
            wait = 2 * attempt
            print(f"  HTTPError {e.code} on '{title}' ({attempt}/{max_retries}), {wait}s待機")
            time.sleep(wait)
        except Exception as e:
            last_err = e
            wait = 2 * attempt
            print(f"  Error on '{title}': {e} ({attempt}/{max_retries}), {wait}s待機")
            time.sleep(wait)
    print(f"  ❌ 取得断念: {title} ({last_err})")
    return ""

wiki_titles = [
    "前頭葉", "後頭葉", "ドーパミン受容体", "エピソード記憶", "グルタミン酸",
    "アセチルコリン", "脳腸相関", "セロトニン", "視床下部", "小脳",
    "側頭葉", "頭頂葉", "ニューロン", "アルツハイマー病", "線条体",
    "扁桃体", "海馬", "睡眠", "注意", "シナプス", "グリア細胞",
    "活動電位", "大脳皮質", "基底核", "ワーキングメモリ", "長期増強",
    "意識", "統合失調症", "迷走神経", "脳下垂体", "副腎皮質ホルモン",
    "抗うつ薬", "ストレス (生体)", "感情", "コルチゾール", "アミノ酸", "ペプチド",
    "イオンチャネル", "事象関連電位", "近赤外線分光法", "γ-アミノ酪酸",
    "ストループ効果", "神経可塑性", "第二言語習得", "失語症",
]

texts_B = []
for title in wiki_titles:
    safe = title.replace("/", "_")
    cache = TMP / f"wiki_{safe}.txt"
    if cache.exists() and cache.stat().st_size > 0:
        t = cache.read_text(encoding="utf-8")
        print(f" cached B: {title}")
    else:
        print(f" downloading B: {title}")
        t = clean_wikipedia_text(fetch_wikipedia_text(title))
        cache.write_text(t, encoding="utf-8")
        time.sleep(1.0)
    if t:
        texts_B.append(f"【{title}】\n{t}")
        print(f"  → {len(t):,} 文字")
    else:
        print(f"  ⚠️ 空/失敗: {title}")

data_B = "\n\n".join(texts_B)
print(f"✅ コーパスB: {len(data_B):,} 文字")



=== 4-2. コーパスB（Wikipedia）===
 cached B: 前頭葉
  → 2,134 文字
 cached B: 後頭葉
  → 1,844 文字
 cached B: ドーパミン受容体
  → 983 文字
 cached B: エピソード記憶
  → 4,238 文字
 cached B: グルタミン酸
  → 3,658 文字
 cached B: アセチルコリン
  → 1,246 文字
 cached B: 脳腸相関
  → 1,392 文字
 cached B: セロトニン
  → 2,741 文字
 cached B: 視床下部
  → 1,252 文字
 cached B: 小脳
  → 15,270 文字
 cached B: 側頭葉
  → 269 文字
 cached B: 頭頂葉
  → 1,743 文字
 cached B: ニューロン
  → 18,139 文字
 cached B: アルツハイマー病
  → 32,741 文字
 cached B: 線条体
  → 3,619 文字
 cached B: 扁桃体
  → 4,144 文字
 cached B: 海馬
  → 599 文字
 cached B: 睡眠
  → 15,789 文字
 cached B: 注意
  → 845 文字
 cached B: シナプス
  → 2,953 文字
 cached B: グリア細胞
  → 1,366 文字
 cached B: 活動電位
  → 7,407 文字
 cached B: 大脳皮質
  → 2,075 文字
 cached B: 基底核
  → 3,033 文字
 downloading B: ワーキングメモリ
  → 6,911 文字
 downloading B: 長期増強
  → 14,944 文字
 cached B: 意識
  → 14,358 文字
 cached B: 統合失調症
  → 36,002 文字
 cached B: 迷走神経
  → 2,197 文字
 cached B: 脳下垂体
  → 1,712 文字
 cached B: 副腎皮質ホルモン
  → 2,494 文字
 cached B: 抗うつ薬
  → 24,361 文字
 cached B: ストレス (生体)


In [ ]:
# ============================================================
# 4-3. 保存
# ============================================================
data_AB = data_A + "\n\n" + data_B
(DIR_A / "input.txt").write_text(data_A, encoding="utf-8")
(DIR_B / "input.txt").write_text(data_B, encoding="utf-8")
(DIR_AB / "input.txt").write_text(data_AB, encoding="utf-8")
print(f"✅ input.txt 保存 A={len(data_A):,} B={len(data_B):,} AB={len(data_AB):,}")


✅ input.txt 保存 A=374,985 B=297,269 AB=672,256


In [ ]:
# ============================================================
# 4-4. 共有 SentencePiece + train/val bin
# ============================================================
print("\n=== 4-4. tokenizer & bins ===")
vocab_size = 8000
spm_prefix = str(DIR_AB / "spm")
spm_model = DIR_AB / "spm.model"

if not spm_model.exists():
    print("--- SentencePiece を A+B で学習 ---")
    spm.SentencePieceTrainer.train(
        input=str(DIR_AB / "input.txt"),
        model_prefix=spm_prefix,
        vocab_size=vocab_size,
        model_type="bpe",
        character_coverage=0.9995,
        bos_id=1, eos_id=2, unk_id=0, pad_id=3,
    )
    print("✅ tokenizer 作成")
else:
    print("✅ tokenizer 既存")

sp = spm.SentencePieceProcessor()
sp.load(str(spm_model))

def save_bins(text: str, out_dir: pathlib.Path, tag: str):
    ids = sp.encode(text, out_type=int)
    n = len(ids)
    print(f"✅ [{tag}] tokens={n:,}")
    split = max(1, int(n * 0.9))
    np.array(ids[:split], dtype=np.uint16).tofile(out_dir / "train.bin")
    np.array(ids[split:], dtype=np.uint16).tofile(out_dir / "val.bin")
    meta = {
        "vocab_size": int(sp.get_piece_size()),
        "sp_model_path": str(spm_model),
        "tag": tag,
        "num_tokens": n,
    }
    with open(out_dir / "meta.pkl", "wb") as f:
        pickle.dump(meta, f)

    # 共有spmを A/B 側へコピー（AB自身への自己コピーはしない）
    for fname in ["spm.model", "spm.vocab"]:
        src = (DIR_AB / fname).resolve()
        dst = (out_dir / fname).resolve()
        if src.exists() and src != dst:
            shutil.copy2(src, dst)

    print(f"✅ [{tag}] -> {out_dir}")

save_bins(data_A, DIR_A, "A")
save_bins(data_B, DIR_B, "B")
save_bins(data_AB, DIR_AB, "A+B")

# ここで初めて存在確認
assert spm_model.exists()
assert (DIR_A / "train.bin").exists()
assert (DIR_B / "train.bin").exists()
print("\n✅ データ準備完了")
print("  ", DIR_A)
print("  ", DIR_B)
print("  ", DIR_AB)
print("  vocab_size =", sp.get_piece_size())




=== 4-4. tokenizer & bins ===
✅ tokenizer 既存
✅ [A] tokens=222,673
✅ [A] -> data/nakajimaton_aozora_bpe
✅ [B] tokens=160,668
✅ [B] -> data/neuro_wiki_bpe
✅ [A+B] tokens=383,341
✅ [A+B] -> data/nakajimaton_neuro_bpe

✅ データ準備完了
   data/nakajimaton_aozora_bpe
   data/neuro_wiki_bpe
   data/nakajimaton_neuro_bpe
  vocab_size = 8000


In [ ]:
# ===== 5. Phase1: Aのみ学習（新規・拡大コーパス）=====
import os
import pathlib

os.chdir("/content/nanoGPT")

OUT_A = "out-A"

!python train.py config/train_gpt2.py \
  --dataset=nakajimaton_aozora_bpe \
  --out_dir=out-A \
  --init_from=scratch \
  --device=cuda \
  --compile=False \
  --gradient_accumulation_steps=1 \
  --eval_interval=200 \
  --eval_iters=20 \
  --log_interval=20 \
  --max_iters=5000 \
  --lr_decay_iters=5000 \
  --warmup_iters=200 \
  --block_size=128 \
  --batch_size=16 \
  --n_layer=4 \
  --n_head=4 \
  --n_embd=128 \
  --dropout=0.1 \
  --wandb_log=False

assert pathlib.Path(OUT_A, "ckpt.pt").exists()
print("✅ Phase1 完了")

Overriding config with config/train_gpt2.py:
# config for training GPT-2 (124M) down to very nice loss of ~2.85 on 1 node of 8X A100 40GB
# launch as the following (e.g. in a screen session) and wait ~5 days:
# $ torchrun --standalone --nproc_per_node=8 train.py config/train_gpt2.py

wandb_log = True
wandb_project = 'owt'
wandb_run_name='gpt2-124M'

# these make the total batch size be ~0.5M
# 12 batch size * 1024 block size * 5 gradaccum * 8 GPUs = 491,520
batch_size = 12
block_size = 1024
gradient_accumulation_steps = 5 * 8

# this makes total number of tokens be 300B
max_iters = 600000
lr_decay_iters = 600000

# eval stuff
eval_interval = 1000
eval_iters = 200
log_interval = 10

# weight decay
weight_decay = 1e-1

Overriding: dataset = nakajimaton_aozora_bpe
Overriding: out_dir = out-A
Overriding: init_from = scratch
Overriding: device = cuda
Overriding: compile = False
Overriding: gradient_accumulation_steps = 1
Overriding: eval_interval = 200
Overriding: eval_iters = 20
Overriding

In [ ]:
# ============================================================
# 6. 評価関数 + A直後評価
# DIR_A, DIR_B, SPM, PROMPTS_* は上のセルの変数を流用
# Aプロンプは中島敦っぽい文語・漢文調でありA学習が効いている
# Bプロンプトは文学調（伯父・孔子・衛など）だが、まだBで学習していないので、B用語は出てこない
# ============================================================

from model import GPTConfig, GPT
from contextlib import nullcontext

sp = spm.SentencePieceProcessor()
sp.load(str(DIR_AB / "spm.model"))

def load_model(ckpt_path, device=device):
    ckpt = torch.load(ckpt_path, map_location=device)
    conf = GPTConfig(**ckpt["model_args"])
    model = GPT(conf)
    state = ckpt["model"]
    for k in list(state.keys()):
        if k.startswith("_orig_mod."):
            state[k[len("_orig_mod."):]] = state.pop(k)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

@torch.no_grad()
def estimate_val_loss(model, data_dir, block_size=128, batch_size=16, eval_iters=20):
    data = np.memmap(pathlib.Path(data_dir) / "val.bin", dtype=np.uint16, mode="r")
    if len(data) <= block_size + 1:
        return float("nan"), float("nan")
    losses = []
    ctx = nullcontext() if device == "cpu" else torch.amp.autocast(device_type="cuda", dtype=torch.float16)
    for _ in range(eval_iters):
        ix = torch.randint(len(data) - block_size - 1, (batch_size,))
        x = torch.stack([torch.from_numpy(data[i:i+block_size].astype(np.int64)) for i in ix]).to(device)
        y = torch.stack([torch.from_numpy(data[i+1:i+1+block_size].astype(np.int64)) for i in ix]).to(device)
        with ctx:
            _, loss = model(x, y)
        losses.append(loss.item())
    mean_loss = sum(losses) / len(losses)
    return mean_loss, math.exp(min(mean_loss, 20))

@torch.no_grad()
def generate_texts(model, prompts, max_new_tokens=80):
    outs = []
    for p in prompts:
        x = torch.tensor([sp.encode(p)], dtype=torch.long, device=device)
        y = model.generate(x, max_new_tokens=max_new_tokens, temperature=0.8, top_k=50)
        outs.append(sp.decode(y[0].tolist()))
    return outs

def evaluate_checkpoint(tag, ckpt_path):
    print("\n" + "=" * 60)
    print("評価:", tag, ckpt_path)
    model = load_model(ckpt_path)
    loss_A, ppl_A = estimate_val_loss(model, DIR_A)
    loss_B, ppl_B = estimate_val_loss(model, DIR_B)
    print(f"  A val loss={loss_A:.4f} ppl≈{ppl_A:.2f}")
    print(f"  B val loss={loss_B:.4f} ppl≈{ppl_B:.2f}")
    print("  --- Aプロンプト ---")
    for t in generate_texts(model, PROMPTS_A):
        print(" >", t[:180].replace("\n", " / "))
    print("  --- Bプロンプト ---")
    for t in generate_texts(model, PROMPTS_B):
        print(" >", t[:180].replace("\n", " / "))
    return dict(tag=tag, loss_A=loss_A, ppl_A=ppl_A, loss_B=loss_B, ppl_B=ppl_B)

results = [evaluate_checkpoint("After A only", f"{OUT_A}/ckpt.pt")]


評価: After A only out-A/ckpt.pt
number of parameters: 1.81M
  A val loss=5.9808 ppl≈395.74
  B val loss=9.7940 ppl≈17925.75
  --- Aプロンプト ---
 > 孔子は思わず言った。 孔※ 子路に向って、叔孫・主人は、まだ魯の使の礼は孔子に降ったい、孔子の北の方々に師に窮していた。臣に窮す。子路に向って、子路はこの罪は無い。公は衛侯に窮死した。孔子が孔子に決めたというよりは、孔子にはこの国を説が無い。ただこの国の殺された。
 > 漢の武帝の終って、李陵が己が死した。ただ北辺に、漢はたまたま浚稽山の谷を ⁇ の都尉・木に陵に向かって帰るのである。漢軍のときとして、李陵を極めたいわっていたが、陵を呼ばれた。漢江の軍隊が、天上は胡兵を失ったと陵との戦した単于の庭で、胡軍
 > 今から一年程前、彼の従姉豹は、殊にその家族よりも更に、独逸人のいない衛律について、自ら大軍に対する所長の都都尉韓には武器で、独逸領から大規準迄は、その父親の王を納得たのである。もちろん、白人の大国は、自分の地にあってその首を取った。 其の妻子の前に使いが、三年の
  --- Bプロンプト ---
 > 前頭葉は、この道具の氷のあたりを思いてしまった。 その晩、伯父自身の歩も、その近通りの世の中にある。 伯父の遺稿を、高等学校の経って、三造の伯父の遺言の実務的な本能的な、伯父がその間も持ち込んだ伯父をもたれた伯父もいなかった。伯父はその「「この伯父であり、自分」の
 > ドーパミン受容体は、天才の秘かに三千、三人の未定の ⁇ の詩の策士、常に、且つ旅に「百呎に ⁇ 図ちん。」 事実の間には、先輩の役の国内に出たぬ。 袁※、彼は、我が国が、彼の一言の心からの使が、この三年の元の主君に、
 > 線条体は、やがて、この風邪は、その手紙を取って、そのために、そのために、その家には、家庭の従姉の主人は土人達の伯姫が都合った。公学校を奪われた一人の老が、その夫婦の諸諸家の前まで来た妻(この祖父)が来た。殊に、という者が、という者が、母父、


In [ ]:
# ============================================================
# Bで追加学習
# 7. Phase2: A→B 継続学習
# ============================================================

import shutil
from pathlib import Path

os.chdir("/content/nanoGPT")

OUT_A = "out-A"
OUT_A_THEN_B = "out-A-then-B"

src = Path(OUT_A) / "ckpt.pt"
dst_dir = Path(OUT_A_THEN_B)
dst_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(src, dst_dir / "ckpt.pt")
print("copied:", src, "->", dst_dir / "ckpt.pt")

# B追加学習
!python train.py config/train_gpt2.py \
  --dataset=neuro_wiki_bpe \
  --out_dir=out-A-then-B \
  --init_from=resume \
  --device=cuda \
  --compile=False \
  --gradient_accumulation_steps=1 \
  --eval_interval=200 \
  --eval_iters=20 \
  --log_interval=20 \
  --max_iters=7000 \
  --lr_decay_iters=7000 \
  --warmup_iters=200 \
  --block_size=128 \
  --batch_size=16 \
  --n_layer=4 \
  --n_head=4 \
  --n_embd=128 \
  --dropout=0.1 \
  --wandb_log=False

print("✅ Phase2 完了")


copied: out-A/ckpt.pt -> out-A-then-B/ckpt.pt
Overriding config with config/train_gpt2.py:
# config for training GPT-2 (124M) down to very nice loss of ~2.85 on 1 node of 8X A100 40GB
# launch as the following (e.g. in a screen session) and wait ~5 days:
# $ torchrun --standalone --nproc_per_node=8 train.py config/train_gpt2.py

wandb_log = True
wandb_project = 'owt'
wandb_run_name='gpt2-124M'

# these make the total batch size be ~0.5M
# 12 batch size * 1024 block size * 5 gradaccum * 8 GPUs = 491,520
batch_size = 12
block_size = 1024
gradient_accumulation_steps = 5 * 8

# this makes total number of tokens be 300B
max_iters = 600000
lr_decay_iters = 600000

# eval stuff
eval_interval = 1000
eval_iters = 200
log_interval = 10

# weight decay
weight_decay = 1e-1

Overriding: dataset = neuro_wiki_bpe
Overriding: out_dir = out-A-then-B
Overriding: init_from = resume
Overriding: device = cuda
Overriding: compile = False
Overriding: gradient_accumulation_steps = 1
Overriding: eval_interval 

In [ ]:
# ============================================================
# 8. 結果比較
# Bへの適合とAの劣化が同時に出ており、古典的な Catastrophic Forgetting のパターンを示している
# A プロンプトは崩れ、一部に「意識」など B 側の語が混ざる
# ============================================================
results.append(evaluate_checkpoint("After A→B", f"{OUT_A_THEN_B}/ckpt.pt"))

print("\n【比較】")
print(f"{'stage':<16} {'loss_A':>10} {'ppl_A':>10} {'loss_B':>10} {'ppl_B':>10}")
for r in results:
    print(f"{r['tag']:<16} {r['loss_A']:10.4f} {r['ppl_A']:10.2f} {r['loss_B']:10.4f} {r['ppl_B']:10.2f}")

dA = results[1]["loss_A"] - results[0]["loss_A"]
dB = results[1]["loss_B"] - results[0]["loss_B"]
print(f"\nΔ loss_A = {dA:+.4f}  (プラスならA悪化＝忘却の目安)")
print(f"Δ loss_B = {dB:+.4f}  (マイナスならB適応)")


評価: After A→B out-A-then-B/ckpt.pt
number of parameters: 1.81M
  A val loss=7.1656 ppl≈1294.14
  B val loss=5.9960 ppl≈401.81
  --- Aプロンプト ---
 > 孔子は思わず始めて子路亡命長老――故顔を虎虎季窮己の顔を椰子荘故虎湖椰子叛独り疲れ虎椰子其の私を欺故部落白人此の駈故其の――私は趙猛牧附ギラ此の猛衛侯大夫子路の※猛李陵は始めて二人の此の此の椰子※二人附其の――猛虎其の其の其の此の此のだけは此の男が喰猛彼等小舎此の此の此の子路※単于彼等
 > 漢の武帝のギラ魯霊公亡命叛ラウペパ長老部落故衛侯衛侯※※リメイギラ此の椰子虎子路は此の牧其の故虎長老其の彼等此の此の喰其の其の顔を故之に附附附衛の趙其の此の喀サモアの部落亡命其の之を二人李陵は彼らの虎己のサモアの虎其の其の私を私は二人の此の子路趙此の此のに於故此の男が故一寸漸く――李陵は魯リメイ叛此の此の其の此の
 > 今から一年程前にも見ない見であるため、他の人では、このようなことは異なったりするのは、その場合、人間の心理を否定できない。これは、その場合、その意味が適用されている。自分にはよくある場合、その問題の瞬間には、その場合、その考えの見つながると、このような場合、このような考えがある、意味を「意識」という。彼らは「意識は
  --- Bプロンプト ---
 > 前頭葉は、この信号に対しても発生することで、NaV、特に大きなもの。 より様々な現象として機能が異なる。 その電気信号は、この際の異常を必要とする状態がある。 他学的な機能は、エピソード記憶を調節し、扁桃体に、神経細胞は、その場合に、これらの刺激、感情が直接的に変化させる。これらの疾患としては、意識には、記憶、多くの
 > ドーパミン受容体はシナプスの膜、グルタミン酸に大きく位置することが知られており、脳・シナプスでの構成されている。 長期増強の膜の分泌による脳の電位 皮質が、歯状核は、神経細胞、神経細胞の入力と、内部の神経伝達物質が主に神経溝の出力が膜から細胞に伝わる。 上小脳脚がよく分岐する。 体管節は、内部に位置する内部に
 > 線条体は、シナプス後細胞が、細胞体に位置する。 軸索(細胞) 小脳細胞・内核(細胞) 内